## 🇺🇦 Wartime Civilian Harms and Adolescent Trauma in Ukraine

_WIP - NOT FOR DISTRIBUTION_

**Preregistration and STROBE checklist in progress**

*Provided in the spirit of open science. This notebook offers a "one-click" replication wrangling and building the _oblast_ (region) and _raion_ (district) Bellingcat OSINT [Civilian Harm in Ukraine](https://ukraine.bellingcat.com/) geocoded event-level data $\rightarrow$ Ukraine Longitudinal Study (ULS) and (non-replicable) individual-level _Ukraine Longitudinal Survey_ (ULS) 1:$n$ merge. To protect the safety and anonymity of the child-adolescent ULS respondents, those data are private and cannot be released. 

⛏️ `uls_scratchpad.ipynb`<br>
Simone J. Skeen x Claude Code (06-18-2026)

1. [Prepare](#1-prepare)
2. [Import + transform](#2-import--transform-bellingcat-osint-civilian-harm-in-ukraine)


### 1. Prepare
Imports requisite packages; customizes outputs.
***
**Dependencies:** Install via `pip install -r requirements.txt` from project root before running.

In [1]:
%%capture

%pip install -r ../../requirements.txt

In [2]:
# Standard library
import json
import os
import re
import sys
import urllib.request
import warnings
from datetime import datetime
from pathlib import Path
from time import sleep

# Add src to path for local imports
sys.path.insert(0, str(Path.cwd().parent))

# Third-party
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from dotenv import load_dotenv
from geopy.geocoders import Nominatim
from IPython.core.interactiveshell import InteractiveShell
from tqdm.notebook import tqdm

# Local
from mappings import ADMIN_UNIT_TO_OBLAST, RAION_UA_TO_EN

In [3]:
# Env variables
load_dotenv()
BELLINGCAT_API_URL = os.getenv('BELLINGCAT_API_URL')

# Output preferences
InteractiveShell.ast_node_interactivity = 'all'

pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

for category in (FutureWarning, UserWarning):
    warnings.simplefilter(action='ignore', category=category)

In [4]:
%%script false --no-raise-error

# Project directory structure
.
└── ukraine_longitudinal_survey/
    ├── config
    ├── data/
    │   ├── raw/
    │   │   ├── level_1
    │   │   └── level_2
    │   └── processed
    ├── src/
    │   └── notebooks
    └── outputs/
        └── figures

In [5]:
# Set working directory to project root; define data paths
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')

# Inputs subdirectories
DATA_RAW = 'data/raw'
DATA_PROC = 'data/processed'
DATA_LVL1 = f'{DATA_RAW}/level_1'   ### ULS child-adolescent survey data; not for public use
DATA_LVL2 = f'{DATA_RAW}/level_2'   ### Bellingcat geotagged OSINT data

In [6]:
# Bellingcat data source
BELLINGCAT_CSV = 'ukr-civharm-2026-01-09.csv'

# ULS merge params
SURVEY_START_DATE = '2025-04-08' ### earliest observation in ULS survey
DATE_FORMAT_INPUT = '%m/%d/%Y'   ### format in source data (if .CSV fallback)
DATE_FORMAT_ISO = '%Y-%m-%d'     ### ISO 8601 for internal use

# Ukrposhta postcode directory (data.gov.ua)
POSTCODE_7Z = 'zvit-dlia-miu-perelik-poshtovikh-indeksiv-ta-viddilen_08-08-2025-csv.7z'

# Geocoding
NOMINATIM_USER_AGENT = 'ukraine_postcode_geocoder'
NOMINATIM_DELAY_SEC = 1  ### 1-second delay; ensures rate limit compliance

In [7]:
%%script false --no-raise-error
#############################################################################################

# Fetch routinely updated .JSON from Bellingcat API endpoint

### docs: https://github.com/bellingcat/ukraine-timemap

def fetch_bellingcat_json(url):
    """
    Fetches Bellingcat civilian harm data from API endpoint.
    Returns list of event dictionaries.
    """
    with urllib.request.urlopen(url) as response:
        data = json.loads(response.read().decode('utf-8'))
    return data

def convert_to_csv_format(events):
    """
    Converts Bellingcat API JSON to CSV format matching ukr-civharm-*.csv structure.
    
    JSON format: id, date (YYYY-MM-DD), latitude, longitude, location, 
                 description, sources (array), impact (array), weapon_system (array)
    CSV format:  id, date (MM/DD/YYYY), latitude, longitude, location,
                 description, sources (comma-sep), associations (formatted string)
    """
    rows = []
    for event in events:
        # Convert date: YYYY-MM-DD → MM/DD/YYYY
        date_iso = event.get('date', '')
        try:
            date_obj = datetime.strptime(date_iso, '%Y-%m-%d')
            date_formatted = date_obj.strftime('%m/%d/%Y')
        except ValueError:
            date_formatted = date_iso
        
        # Join sources array
        sources = event.get('sources', [])
        sources_str = ','.join(sources) if sources else ''
        
        # Build `associations` string from `impact` & `weapon_system`
        associations_parts = []
        for impact in event.get('impact', []):
            associations_parts.append(f'Type of area affected={impact}')
        for weapon in event.get('weapon_system', []):
            associations_parts.append(f'Weapon System={weapon}')
        associations_str = ','.join(associations_parts) if associations_parts else ''
        
        rows.append({
            'id': event.get('id', ''),
            'date': date_formatted,
            'latitude': event.get('latitude', ''),
            'longitude': event.get('longitude', ''),
            'location': event.get('location', '').strip(),
            'description': event.get('description', '').strip(),
            'sources': sources_str,
            'associations': associations_str,
        })
    
    return pd.DataFrame(rows)

# Fetch
print(f"Fetching data from Bellingcat API...")
d_lvl2_raw_json = fetch_bellingcat_json(BELLINGCAT_API_URL)

# Save raw .JSON 
json_path = f'{DATA_LVL2}/ukr-civharm-{datetime.now().strftime('%Y-%m-%d')}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(d_lvl2_raw_json, f, ensure_ascii=False, indent=2)
print(f"Raw JSON saved to: {json_path}")

# Convert & save to .CSV
d_lvl2_raw_csv = convert_to_csv_format(d_lvl2_raw_json)
csv_path = f'{DATA_LVL2}/ukr-civharm-{datetime.now().strftime('%Y-%m-%d')}.csv'
d_lvl2_raw_csv.to_csv(csv_path, index=False)

print(f"Fetched {len(d_lvl2_raw_csv):,} events")
print(f"Saved to: {csv_path}")
d_lvl2_raw_csv.head(3)
#############################################################################################

### 2. Import + transform: Bellingcat OSINT Civilian Harm in Ukraine
Imports, cleans, describes level-2 aggregate conflict data. Acquired via .csv export: https://ukraine.bellingcat.com/.

In [8]:
#############################################################################################
d = pd.read_csv(f'data/ukr-civharm-2026-01-09.csv')
d.head(5)
#############################################################################################

,id,date,latitude,longitude,location,description,sources,associations
0,CIV0098,02/24/2022,49.212119,38.905921,NaN,"Individual injured by shelling, ambulance resp...",https://www.facebook.com/story.php?story_fbid=...,"Type of area affected=Residential,Weapon Syste..."
1,CIV0013,02/24/2022,48.055395,37.778300,NaN,Apparent strike on hospital in separatist held...,https://twitter.com/City_Donetsk/status/149687...,"Type of area affected=Healthcare,Weapon System..."
2,CIV0004,02/24/2022,47.775609,37.239673,NaN,"Explosion in central Kyiv, nothing further yet.",https://twitter.com/N_Waters89/status/14968566...,"Type of area affected=Healthcare,Weapon System..."
3,CIV0011,02/24/2022,50.308310,34.880702,NaN,Civilian buildings damaged and destroyed by sh...,https://t.me/nexta_live/16696,"Type of area affected=Commercial,Weapon System..."
4,CIV0008,02/24/2022,47.117099,37.684831,NaN,civilian homes hit by artillery.,"https://t.me/mariupolrada/8465,https://twitter...","Type of area affected=Residential,Weapon Syste..."


In [9]:
# Dupe raw for processing
#d = d_lvl2_raw_csv.copy()

# Add ascending numerical index
d['index'] = range(len(d))
d = d.set_index('index')

# Drop imprecise OS location col
d = d.drop(
    'location', 
    axis = 1, 
    errors = 'ignore',
    )

# Restrict to obs on or before ULS start date
d['date'] = pd.to_datetime(
    d['date'], 
    format = DATE_FORMAT_INPUT,
    errors = 'coerce',    
    )

uls_startdate = pd.to_datetime(SURVEY_START_DATE)
d = d[d['date'] <= uls_startdate]

# Inspect & verify
d.shape
d.info()
d.head(2)
d.tail(2)

(2446, 7)

<class 'pandas.core.frame.DataFrame'>
Index: 2446 entries, 0 to 2445
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id            2446 non-null   object        
 1   date          2446 non-null   datetime64[ns]
 2   latitude      2444 non-null   float64       
 3   longitude     2444 non-null   float64       
 4   description   2446 non-null   object        
 5   sources       2445 non-null   object        
 6   associations  2405 non-null   object        
dtypes: datetime64[ns](1), float64(2), object(4)
memory usage: 152.9+ KB


,id,date,latitude,longitude,description,sources,associations
index,,,,,,,
0,CIV0098,2022-02-24,49.212119,38.905921,"Individual injured by shelling, ambulance resp...",https://www.facebook.com/story.php?story_fbid=...,"Type of area affected=Residential,Weapon Syste..."
1,CIV0013,2022-02-24,48.055395,37.778300,Apparent strike on hospital in separatist held...,https://twitter.com/City_Donetsk/status/149687...,"Type of area affected=Healthcare,Weapon System..."


,id,date,latitude,longitude,description,sources,associations
index,,,,,,,
2444,S23173,2025-04-07,48.507810,37.747111,At least one person killed and two injured inc...,"https://t.me/astrapress/78387,https://t.me/ast...","Type of area affected=Residential,Type of area..."
2445,W4DR1R,2025-04-08,50.775975,35.251465,Heavily damaged residential buildings followin...,"https://t.me/suspilnesumy/32400,https://t.me/s...",Type of area affected=Residential


In [10]:
# Dummy code: area type affected & weapon system

### Creates binary indicators for all `associations` values

# === TYPE OF AREA AFFECTED ===
area_types = {
    'admn': 'Administrative',           
    'cmrc': 'Commercial',
    'cltr': 'Cultural',
    'food': 'Food/Food Infrastructure',
    'hlth': 'Healthcare',
    'hmnt': 'Humanitarian',
    'indl': 'Industrial',
    'mltr': 'Military',
    'rlgs': 'Religious',
    'rsdn': 'Residential',
    'trsp': 'Roads/Highways/Transport',
    'schl': 'School or childcare',
    'undf': 'Undefined',
    }

for var, label in area_types.items():
    d[var] = d['associations'].str.contains(
        rf'Type of area affected={re.escape(label)}',
        case=False,
        na=False,
        regex=True,
    ).astype(int)

# === WEAPON SYSTEM ===
weapon_systems = {
    'arst': 'Air strike',
    'amsl': 'Anti-air missile',
    'blsc': 'Ballistic missile',
    'clst': 'Cluster munitions',
    'crsm': 'Cruise missile',
    'mrtr': 'HE artillery inc mortars',
    'rckt': 'HE rocket artillery',
    'tube': 'HE tube artillery',
    'incd': 'Incendiary munitions',
    'mine': 'Land mines',
    'ltrm': 'Loitering munition',
    'none': 'None',
    'smll': 'Small arms',
    'thrm': 'Thermobaric munition',
    'unkn': 'Unknown',
    'vhcl': 'Vehicle mounted weapon',
    }

for var, label in weapon_systems.items():
    d[var] = d['associations'].str.contains(
        rf'Weapon System={re.escape(label)}',
        case=False,
        na=False,
        regex=True,
    ).astype(int)

# Verify counts

print("=== TYPE OF AREA AFFECTED ===")
for var, label in area_types.items():
    print(f"  {var} ({label}): {d[var].sum()}")

print("\n=== WEAPON SYSTEM ===")
for var, label in weapon_systems.items():
    print(f"  {var} ({label}): {d[var].sum()}")

=== TYPE OF AREA AFFECTED ===
  admn (Administrative): 123
  cmrc (Commercial): 395
  cltr (Cultural): 105
  food (Food/Food Infrastructure): 52
  hlth (Healthcare): 139
  hmnt (Humanitarian): 34
  indl (Industrial): 181
  mltr (Military): 7
  rlgs (Religious): 65
  rsdn (Residential): 1094
  trsp (Roads/Highways/Transport): 231
  schl (School or childcare): 326
  undf (Undefined): 71

=== WEAPON SYSTEM ===
  arst (Air strike): 68
  amsl (Anti-air missile): 40
  blsc (Ballistic missile): 47
  clst (Cluster munitions): 117
  crsm (Cruise missile): 82
  mrtr (HE artillery inc mortars): 13
  rckt (HE rocket artillery): 62
  tube (HE tube artillery): 6
  incd (Incendiary munitions): 16
  mine (Land mines): 6
  ltrm (Loitering munition): 111
  none (None): 1
  smll (Small arms): 31
  thrm (Thermobaric munition): 8
  unkn (Unknown): 1300
  vhcl (Vehicle mounted weapon): 16


In [11]:
#############################################################################################
d.head(5)
#############################################################################################

,id,date,latitude,longitude,description,sources,associations,admn,cmrc,cltr,food,hlth,hmnt,indl,mltr,rlgs,rsdn,trsp,schl,undf,arst,amsl,blsc,clst,crsm,mrtr,rckt,tube,incd,mine,ltrm,none,smll,thrm,unkn,vhcl
index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,CIV0098,2022-02-24,49.212119,38.905921,"Individual injured by shelling, ambulance resp...",https://www.facebook.com/story.php?story_fbid=...,"Type of area affected=Residential,Weapon Syste...",0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,CIV0013,2022-02-24,48.055395,37.778300,Apparent strike on hospital in separatist held...,https://twitter.com/City_Donetsk/status/149687...,"Type of area affected=Healthcare,Weapon System...",0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,CIV0004,2022-02-24,47.775609,37.239673,"Explosion in central Kyiv, nothing further yet.",https://twitter.com/N_Waters89/status/14968566...,"Type of area affected=Healthcare,Weapon System...",0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
3,CIV0011,2022-02-24,50.308310,34.880702,Civilian buildings damaged and destroyed by sh...,https://t.me/nexta_live/16696,"Type of area affected=Commercial,Weapon System...",0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,CIV0008,2022-02-24,47.117099,37.684831,civilian homes hit by artillery.,"https://t.me/mariupolrada/8465,https://twitter...","Type of area affected=Residential,Weapon Syste...",0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


#### 2a. Reverse geocode: latitude / longitude $\rightarrow$ UA postcode
Reverse geocodes event coordinates to Ukrainian postcodes via Nominatim API.

In [12]:
#############################################################################################
# Restrict to n = 100 for geocoding tests
d = d.iloc[:100]
d.info()
#############################################################################################

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 36 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id            100 non-null    object        
 1   date          100 non-null    datetime64[ns]
 2   latitude      100 non-null    float64       
 3   longitude     100 non-null    float64       
 4   description   100 non-null    object        
 5   sources       100 non-null    object        
 6   associations  96 non-null     object        
 7   admn          100 non-null    int64         
 8   cmrc          100 non-null    int64         
 9   cltr          100 non-null    int64         
 10  food          100 non-null    int64         
 11  hlth          100 non-null    int64         
 12  hmnt          100 non-null    int64         
 13  indl          100 non-null    int64         
 14  mltr          100 non-null    int64         
 15  rlgs          100 non-null    int64         
 

In [13]:
# Convert latitude/longitude coordinates → Ukrainian postcodes via Nominatim API
geolocator = Nominatim(user_agent = NOMINATIM_USER_AGENT)

def get_postcode(lat, lon):
    """
    Reverse geocode latitude/longitude to get postcode.
    Returns None if postcode not found.
    """
    try:
        location = geolocator.reverse(f"{lat}, {lon}", language = 'en')
        if location and location.raw.get('address'):
            postcode = location.raw['address'].get('postcode')
            return postcode
        return None
    except Exception as e:
        print(f"Error geocoding ({lat}, {lon}): {e}")
        return None

# Apply geocoding to each row with rate-limited delay
postcodes = []

for idx, row in tqdm(d.iterrows(), total=len(d), desc="Geocoding postcodes"):
    lat = row['latitude']
    lon = row['longitude']
    
    postcode = get_postcode(lat, lon)
    postcodes.append(postcode)
    
    sleep(NOMINATIM_DELAY_SEC)

d['postcode'] = postcodes

print(f"\nGeocoding complete!")
print(f"Postcodes found: {d['postcode'].notna().sum()}/{len(d)}")
print(f"\nSample results:")
print(d[['latitude', 'longitude', 'postcode']].head(10))

Geocoding postcodes:   0%|          | 0/100 [00:00<?, ?it/s]


Geocoding complete!
Postcodes found: 99/100

Sample results:
        latitude  longitude postcode
index                               
0      49.212119  38.905921    92760
1      48.055395  37.778300    83054
2      47.775609  37.239673    85670
3      50.308310  34.880702    42700
4      47.117099  37.684831    87503
5      46.227914  34.642854    75565
6      48.748938  30.218889    20300
7      49.850781  36.659762    63501
8      48.051489  37.778075    83054
9      47.379841  37.756229    87100


In [14]:
# Enumerate unique postcodes in d
def count_unique_postcodes(df, col = 'postcode'):
    """
    Returns count of unique non-null values in specified column.
    """
    unique_vals = df[col].dropna().unique()
    return len(unique_vals)

n_unique = count_unique_postcodes(d)
print(f"Unique postcodes in d: {n_unique}")

Unique postcodes in d: 76


In [15]:
# Extract first 2 digits of postcode as `admin_unit` id
d['admin_unit'] = d['postcode'].astype(str).str[:2]

# Replace 'na' (from NaN conversion) with actual NaN
d.loc[d['admin_unit'] == 'na', 'admin_unit'] = np.nan

# Enumerate unique admin units
n_unique_admin = count_unique_postcodes(
    d, 
    col = 'admin_unit',
    )
print(f"Unique admin units in d: {n_unique_admin}")
print(f"\nAdmin unit distribution:")
d['admin_unit'].value_counts().sort_index()

Unique admin units in d: 26

Admin unit distribution:


admin_unit
01     1
02     1
03     1
04     5
07     2
08    11
14     4
15     1
20     1
34     1
42     4
54     1
61    28
62     2
63     2
72     1
73     2
74     2
75     1
83     6
84     3
85     5
87     6
92     4
93     4
No     1
Name: count, dtype: int64

In [16]:
# Map `admin_unit` → `oblast`
### Uses ADMIN_UNIT_TO_OBLAST from mappings.py
### Source: Ukrposhta postal code system: https://en.wikipedia.org/wiki/Postal_codes_in_Ukraine

d['oblast'] = d['admin_unit'].map(ADMIN_UNIT_TO_OBLAST)

# Verify mapping
print(f"Mapped oblasts: {d['oblast'].notna().sum()}/{len(d)}")
print(f"Unmapped admin units: {d[d['oblast'].isna()]['admin_unit'].unique()}")
print(f"\nOblast distribution:")
d['oblast'].value_counts()

Mapped oblasts: 99/100
Unmapped admin units: ['No']

Oblast distribution:


oblast
Kharkiv Oblast         32
Donetsk Oblast         20
Kyiv Oblast            13
Luhansk Oblast          8
Kyiv                    8
Kherson Oblast          5
Chernihiv Oblast        5
Sumy Oblast             4
Cherkasy Oblast         1
Zaporizhzhia Oblast     1
Rivne Oblast            1
Mykolaiv Oblast         1
Name: count, dtype: int64

#### PRELIM: Encode _raions_ two ways
**Note:** Ukrposhta postcode lookup will not return Russian-occupied districts. These are denoted `<Rus-occupied>` in `raion_postcode`:<br>

|Prefix|Region|$n$ raions|
|------|------|------------------|
|83xxx|Donetsk city|0 (`<Rus-occupied>`)|
|91xxx|Luhansk city|0 (`<Rus-occupied>`)|
|94xxx|Luhansk oblast|0 (`<Rus-occupied>`)|
|95-99|Crimea/Sevastopol|0 (`<Rus-occupied>`)|

In [17]:
# Map latitude/longitude coordinates → Ukrainian raions via Nominatim API
### Source: ukraine_raion_lookup.py (option 4)
### For Ukraine, Nominatim returns raion in the "district" field
### Rate-limited: 1 req/sec on public instance; not suitable for bulk geocoding

def raion_from_point_nominatim(lat, lon, user_agent, email=None):
    """
    Reverse geocode lat/lon to Ukrainian raion via OSM Nominatim.
    Returns raion name from 'district' field, or None if not found.
    """
    params = {
        'lat': lat,
        'lon': lon,
        'format': 'jsonv2',
        'addressdetails': 1,
    }
    headers = {'User-Agent': user_agent}
    if email:
        params['email'] = email
    
    try:
        resp = requests.get(
            'https://nominatim.openstreetmap.org/reverse',
            params=params,
            headers=headers,
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()
        sleep(NOMINATIM_DELAY_SEC)  # respect rate limit
        return data.get('address', {}).get('district')
    except Exception as e:
        print(f"Error geocoding ({lat}, {lon}): {e}")
        return None

# Apply to dataframe with progress bar
raions_nominatim = []

for idx, row in tqdm(d.iterrows(), total=len(d), desc="Geocoding raions"):
    raion = raion_from_point_nominatim(
        row['latitude'], 
        row['longitude'], 
        user_agent=NOMINATIM_USER_AGENT,
    )
    raions_nominatim.append(raion)

d['raion_nominatim_ua'] = raions_nominatim

print(f"\nRaion geocoding complete!")
print(f"Raions found: {d['raion_nominatim_ua'].notna().sum()}/{len(d)}")
print(f"\nSample results:")
d[['latitude', 'longitude', 'raion_nominatim_ua']].head(10)

Geocoding raions:   0%|          | 0/100 [00:00<?, ?it/s]


Raion geocoding complete!
Raions found: 92/100

Sample results:


,latitude,longitude,raion_nominatim_ua
index,,,
0,49.212119,38.905921,Старобільський район
1,48.055395,37.778300,Донецький район
2,47.775609,37.239673,Волноваський район
3,50.308310,34.880702,Охтирський район
4,47.117099,37.684831,Маріупольський район
5,46.227914,34.642854,Генічеський район
6,48.748938,30.218889,Уманський район
7,49.850781,36.659762,Чугуївський район
8,48.051489,37.778075,Донецький район


In [18]:
# Map Ukrainian raion names → English translations
### Uses RAION_UA_TO_EN from mappings.py
### Source: https://en.wikipedia.org/wiki/Raions_of_Ukraine (post-2020 reform: 136 raions)

d['raion_nominatim_en'] = d['raion_nominatim_ua'].map(RAION_UA_TO_EN)

# Report coverage
mapped = d['raion_nominatim_en'].notna().sum()
total_with_ua = d['raion_nominatim_ua'].notna().sum()
print(f"Mapped to English: {mapped}/{total_with_ua}")

# Show any unmapped Ukrainian raions for dictionary updates
unmapped = d[d['raion_nominatim_ua'].notna() & d['raion_nominatim_en'].isna()]['raion_nominatim_ua'].unique()
if len(unmapped) > 0:
    print(f"\nUnmapped raions (add to mappings.py):")
    for r in unmapped:
        print(f"    '{r}': '',")

print(f"\nSample results:")
d[['raion_nominatim_ua', 'raion_nominatim_en']].head(10)

Mapped to English: 88/92

Unmapped raions (add to mappings.py):
    'Сіверськодонецький район': '',

Sample results:


,raion_nominatim_ua,raion_nominatim_en
index,,
0,Старобільський район,Starobilsk Raion
1,Донецький район,Donetsk Raion
2,Волноваський район,Volnovakha Raion
3,Охтирський район,Okhtyrka Raion
4,Маріупольський район,Mariupol Raion
5,Генічеський район,Henichesk Raion
6,Уманський район,Uman Raion
7,Чугуївський район,Chuhuiv Raion
8,Донецький район,Donetsk Raion


In [19]:
# Extract postcode directory from .7z archive
import py7zr

POSTCODE_7Z_PATH = f'{DATA_RAW}/{POSTCODE_7Z}'

# Extract if .7z exists
if os.path.exists(POSTCODE_7Z_PATH):
    # Check if already extracted by looking for any CSV with Ukrainian postal keywords
    existing_csvs = [f for f in os.listdir(DATA_RAW) if f.endswith('.csv') and 'індекс' in f.lower()]
    
    if not existing_csvs:
        print(f"Extracting {POSTCODE_7Z_PATH}...")
        with py7zr.SevenZipFile(POSTCODE_7Z_PATH, mode='r') as archive:
            archive.extractall(path=DATA_RAW)
        print(f"Extracted to: {DATA_RAW}/")
        existing_csvs = [f for f in os.listdir(DATA_RAW) if f.endswith('.csv') and 'індекс' in f.lower()]
    
    # Set path to the extracted CSV
    if existing_csvs:
        POSTCODE_DIR_PATH = f'{DATA_RAW}/{existing_csvs[0]}'
        print(f"Postcode directory: {POSTCODE_DIR_PATH}")
    else:
        # Fallback: find any newly created CSV
        all_csvs = [f for f in os.listdir(DATA_RAW) if f.endswith('.csv')]
        print(f"CSV files found: {all_csvs}")
        if all_csvs:
            POSTCODE_DIR_PATH = f'{DATA_RAW}/{all_csvs[0]}'
            print(f"Using: {POSTCODE_DIR_PATH}")
else:
    print(f"Archive not found: {POSTCODE_7Z_PATH}")
    print("Download from: https://data.gov.ua/dataset/post-index-and-braches")
    POSTCODE_DIR_PATH = None

Postcode directory: data/raw/Звіт для МІУ. Перелік поштових індексів та відділень_08.08.2025.csv


In [20]:
# Map postcodes → Ukrainian raions via Ministry of Community and Territorial Development of Ukraine open data
### Source: ukraine_raion_lookup.py (option 3)
### Data: https://data.gov.ua/dataset/post-index-and-braches
### Note: Column headers are in Ukrainian and may vary between releases; inspect after download

def load_postcode_directory(csv_path):
    """
    Load the data.gov.ua "post-index-and-braches" CSV.
    Tries multiple encodings common for Ukrainian government data.
    Uses semicolon delimiter (European CSV format).
    """
    encodings = ['cp1251', 'windows-1251', 'utf-8', 'iso-8859-5', 'utf-16']
    
    for encoding in encodings:
        try:
            df = pd.read_csv(
                csv_path, 
                encoding=encoding, 
                dtype=str,
                sep=';',            ### European .CSV uses semicolon
                on_bad_lines='skip' ### Skips malformed rows
            )
            print(f"Successfully loaded with encoding: {encoding}")
            return df
        except (UnicodeDecodeError, UnicodeError):
            continue
    
    raise ValueError(f"Could not decode {csv_path} with any known encoding")

def raion_from_postcode(postcode, directory, postcode_col, raion_col):
    """
    Direct table lookup for postcode → raion.
    Ukrainian postal codes do NOT cleanly encode raion in digit positions;
    use this authoritative directory rather than parsing the string.
    """
    row = directory.loc[directory[postcode_col] == str(postcode)]
    if row.empty:
        return None
    return row.iloc[0][raion_col]

# Load directory and inspect columns
try:
    postcode_dir = load_postcode_directory(POSTCODE_DIR_PATH)
    print(f"Postcode directory loaded: {len(postcode_dir):,} entries")
    print(f"Columns: {postcode_dir.columns.tolist()}")
    
    # Use English column names from the file
    # Adjust these if your file has different column names
    POSTCODE_COL = 'Postindex VPZ'    ### postcode column
    RAION_COL = 'Distinct (Rayon)'    ### raion column (note: "Distinct" is likely a typo for "District")
    
    # Verify columns exist
    if POSTCODE_COL not in postcode_dir.columns or RAION_COL not in postcode_dir.columns:
        print(f"\nWARNING: Expected columns not found!")
        print(f"Looking for: '{POSTCODE_COL}', '{RAION_COL}'")
        print(f"Available: {postcode_dir.columns.tolist()}")
    else:
        # Apply lookup to dataframe
        d['raion_postcode'] = d['postcode'].apply(
            lambda pc: raion_from_postcode(pc, postcode_dir, POSTCODE_COL, RAION_COL)
        )
        
        # Replace None with <Rus-occupied> (postcodes in occupied territories not in Ukrposhta directory)
        d['raion_postcode'] = d['raion_postcode'].fillna('<Rus-occupied>')
        
        print(f"\nRaions found via postcode: {(d['raion_postcode'] != '<Rus-occupied>').sum()}/{len(d)}")
        print(f"Rus-occupied: {(d['raion_postcode'] == '<Rus-occupied>').sum()}/{len(d)}")
        print(f"\nSample results:")
        d[['postcode', 'raion_postcode']].head(10)
    
except FileNotFoundError:
    print(f"Postcode directory not found at: {POSTCODE_DIR_PATH}")
    print("Download from: https://data.gov.ua/dataset/post-index-and-braches")
    print("Save as 'postindex.7z' in data/raw/ and run extraction cell above")

Successfully loaded with encoding: cp1251
Postcode directory loaded: 320,249 entries
Columns: ['Назва області', 'Назва району', 'Назва населеного пункту (повна)', 'Поштовий індекс населеного пункту', 'Назва вулиці', 'Номер будинку', "Назва відділення зв'язку", "Поштовий індекс відділення зв'язку (ВПЗ)", 'Region (Oblast)', 'Distinct (Rayon)', 'Locality', 'Postindex Locality', 'Street', 'House_numbers', 'Post office', 'Postindex VPZ']

Raions found via postcode: 69/100
Rus-occupied: 31/100

Sample results:


,postcode,raion_postcode
index,,
0,92760,Starobilskyi
1,83054,<Rus-occupied>
2,85670,Volnovaskyi
3,42700,<Rus-occupied>
4,87503,Mariupolskyi
5,75565,<Rus-occupied>
6,20300,<Rus-occupied>
7,63501,Chuhuivskyi
8,83054,<Rus-occupied>


In [21]:
#############################################################################################
# DEBUG: Investigate postcode lookup misses

test_postcode = '83054'
print(f"Investigating postcode: {test_postcode}\n")

# Check both postcode columns in directory
postcode_cols = ['Postindex VPZ', 'Postindex Locality']
for col in postcode_cols:
    if col in postcode_dir.columns:
        exact = postcode_dir[postcode_dir[col] == test_postcode]
        partial = postcode_dir[postcode_dir[col].str.contains(test_postcode, na=False)]
        print(f"'{col}':")
        print(f"  Exact match: {len(exact)} rows")
        print(f"  Partial match: {len(partial)} rows")
        print(f"  Sample values: {postcode_dir[col].dropna().unique()[:10].tolist()}\n")

# Check what postcodes ARE in directory for Donetsk oblast (83-87)
donetsk_postcodes = postcode_dir[postcode_dir['Postindex VPZ'].str.startswith('83', na=False)]
print(f"Donetsk (83xxx) postcodes in directory: {len(donetsk_postcodes)}")
if len(donetsk_postcodes) > 0:
    print(f"Sample: {donetsk_postcodes['Postindex VPZ'].unique()[:10].tolist()}")

# Check Luhansk (91-94)
luhansk_postcodes = postcode_dir[postcode_dir['Postindex VPZ'].str.startswith('92', na=False)]
print(f"\nLuhansk (92xxx) postcodes in directory: {len(luhansk_postcodes)}")
if len(luhansk_postcodes) > 0:
    print(f"Sample: {luhansk_postcodes['Postindex VPZ'].unique()[:10].tolist()}")

# Summary: which oblasts are missing?
print(f"\nOblast prefixes in directory:")
postcode_dir['prefix'] = postcode_dir['Postindex VPZ'].str[:2]
print(postcode_dir['prefix'].value_counts().sort_index())
#############################################################################################

Investigating postcode: 83054

'Postindex VPZ':
  Exact match: 0 rows
  Partial match: 0 rows
  Sample values: ['21001', '21002', '21003', '21004', '21005', '21006', '21007', '21008', '21009', '21010']

'Postindex Locality':
  Exact match: 0 rows
  Partial match: 0 rows
  Sample values: ['21001', '21002', '21003', '21004', '21005', '21006', '21007', '21008', '21009', '21010']

Donetsk (83xxx) postcodes in directory: 0

Luhansk (92xxx) postcodes in directory: 2122
Sample: ['92100', '92110', '92111', '92112', '92114', '92115', '92120', '92121', '92123', '92124']

Oblast prefixes in directory:
prefix
01     299
02     691
03    1112
04     530
07    3888
08    6343
09    3005
10     968
11    2263
12    1615
13    1598
14     641
15    1203
16    1620
17    1632
18     526
19    3089
20    3209
21    1031
22    2458
23    3381
24    2534
25    1021
26    1410
27    1768
28    1268
29     676
30    1285
31    1633
32    2646
33     567
34    2912
35    2798
36    1097
37    2768
38    1239

In [22]:
d.head(5)

,id,date,latitude,longitude,description,sources,associations,admn,cmrc,cltr,food,hlth,hmnt,indl,mltr,rlgs,rsdn,trsp,schl,undf,arst,amsl,blsc,clst,crsm,mrtr,rckt,tube,incd,mine,ltrm,none,smll,thrm,unkn,vhcl,postcode,admin_unit,oblast,raion_nominatim_ua,raion_nominatim_en,raion_postcode
index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,CIV0098,2022-02-24,49.212119,38.905921,"Individual injured by shelling, ambulance resp...",https://www.facebook.com/story.php?story_fbid=...,"Type of area affected=Residential,Weapon Syste...",0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,92760,92,Luhansk Oblast,Старобільський район,Starobilsk Raion,Starobilskyi
1,CIV0013,2022-02-24,48.055395,37.778300,Apparent strike on hospital in separatist held...,https://twitter.com/City_Donetsk/status/149687...,"Type of area affected=Healthcare,Weapon System...",0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,83054,83,Donetsk Oblast,Донецький район,Donetsk Raion,<Rus-occupied>
2,CIV0004,2022-02-24,47.775609,37.239673,"Explosion in central Kyiv, nothing further yet.",https://twitter.com/N_Waters89/status/14968566...,"Type of area affected=Healthcare,Weapon System...",0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,85670,85,Donetsk Oblast,Волноваський район,Volnovakha Raion,Volnovaskyi
3,CIV0011,2022-02-24,50.308310,34.880702,Civilian buildings damaged and destroyed by sh...,https://t.me/nexta_live/16696,"Type of area affected=Commercial,Weapon System...",0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,42700,42,Sumy Oblast,Охтирський район,Okhtyrka Raion,<Rus-occupied>
4,CIV0008,2022-02-24,47.117099,37.684831,civilian homes hit by artillery.,"https://t.me/mariupolrada/8465,https://twitter...","Type of area affected=Residential,Weapon Syste...",0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,87503,87,Donetsk Oblast,Маріупольський район,Mariupol Raion,Mariupolskyi


In [23]:
# Validation: Compare raion mappings from both methods

# Check where both methods returned a result
both_valid = d['raion_nominatim_ua'].notna() & (d['raion_postcode'] != '<Rus-occupied>')
n_both = both_valid.sum()

print(f"Rows with both raion values: {n_both}/{len(d)}")

if n_both > 0:
    # Compare results (case-insensitive, strip whitespace)
    d['raion_match'] = d.apply(
        lambda row: (
            str(row['raion_nominatim_ua']).lower().strip() == 
            str(row['raion_postcode']).lower().strip()
        ) if pd.notna(row['raion_nominatim_ua']) and row['raion_postcode'] != '<Rus-occupied>' else np.nan,
        axis=1
    )
    
    n_match = d['raion_match'].sum()
    match_rate = n_match / n_both * 100 if n_both > 0 else 0
    
    print(f"Exact matches: {int(n_match)}/{n_both} ({match_rate:.1f}%)")
    
    # Show mismatches for inspection
    mismatches = d[both_valid & (d['raion_match'] == False)][
        ['postcode', 'latitude', 'longitude', 'raion_nominatim_ua', 'raion_postcode']
    ]
    if len(mismatches) > 0:
        print(f"\nMismatches ({len(mismatches)}):")
        mismatches.head(10)
    else:
        print("\nNo mismatches found!")

Rows with both raion values: 63/100
Exact matches: 0/63 (0.0%)

Mismatches (63):


,postcode,latitude,longitude,raion_nominatim_ua,raion_postcode
index,,,,,
0,92760,49.212119,38.905921,Старобільський район,Starobilskyi
2,85670,47.775609,37.239673,Волноваський район,Volnovaskyi
4,87503,47.117099,37.684831,Маріупольський район,Mariupolskyi
7,63501,49.850781,36.659762,Чугуївський район,Chuhuivskyi
10,87503,47.120906,37.680990,Маріупольський район,Mariupolskyi
14,08290,50.555440,30.274574,Бучанський район,Buchanskyi
15,08290,50.554625,30.274866,Бучанський район,Buchanskyi
16,61168,50.022621,36.330786,Харківський район,Kharkivskyi
19,74900,46.757458,33.370035,Каховський район,Kakhovskyi


In [24]:
#############################################################################################
# DEBUG: Inspect raw Nominatim response for a sample coordinate
import requests

# Use first row with valid coordinates
sample_row = d[d['latitude'].notna()].iloc[0]
lat, lon = sample_row['latitude'], sample_row['longitude']

print(f"Testing coordinates: ({lat}, {lon})")

params = {
    'lat': lat,
    'lon': lon,
    'format': 'jsonv2',
    'addressdetails': 1,
    }
headers = {'User-Agent': NOMINATIM_USER_AGENT}

resp = requests.get(
    'https://nominatim.openstreetmap.org/reverse',
    params=params,
    headers=headers,
    timeout=10,
    )
data = resp.json()

print(f"\nFull address breakdown:")
for key, value in data.get('address', {}).items():
    print(f"  {key}: {value}")

print(f"\nDisplay name: {data.get('display_name', 'N/A')}")
#############################################################################################

Testing coordinates: (49.212119, 38.905921)

Full address breakdown:
  village: Толоківка
  municipality: Старобільська міська громада
  district: Старобільський район
  state: Луганська область
  ISO3166-2-lvl4: UA-09
  postcode: 92760
  country: Україна
  country_code: ua

Display name: Толоківка, Старобільська міська громада, Старобільський район, Луганська область, 92760, Україна


In [25]:
#Export event-level data prior to aggregation
d.to_csv(f'{DATA_PROC}/d_lvl2_event.csv', index = False) 

### 3. Collapse / export: Bellingcat OSINT Civilian Harm in Ukraine

In [26]:
# Aggregate at raion level

### Collapses event-level data to raion-level counts
### Uses raion_nominatim_en as primary raion identifier

# Define all dummy variables to aggregate
area_type_vars = ['admn', 'cmrc', 'cltr', 'food', 'hlth', 'hmnt', 
                  'indl', 'mltr', 'rlgs', 'rsdn', 'trsp', 'schl', 'undf']

weapon_sys_vars = ['arst', 'amsl', 'blsc', 'clst', 'crsm', 'mrtr',
                   'rckt', 'tube', 'incd', 'mine', 'ltrm', 'none',
                   'smll', 'thrm', 'unkn', 'vhcl']

all_dummy_vars = area_type_vars + weapon_sys_vars

# Build aggregation dict: sum all dummy vars, count events
agg_dict = {var: 'sum' for var in all_dummy_vars}
agg_dict['id'] = 'count'  # count events per raion

# Aggregate by raion (English name)
d_lvl2_raion = d.groupby('raion_nominatim_en', as_index=False).agg(agg_dict)

# Rename id count column
d_lvl2_raion = d_lvl2_raion.rename(columns={'id': 'n_events'})

# Add oblast mapping (most common oblast per raion)
oblast_map = d.groupby('raion_nominatim_en')['oblast'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan
)
d_lvl2_raion['oblast'] = d_lvl2_raion['raion_nominatim_en'].map(oblast_map)

# Reorder columns: raion, oblast, n_events, area types, weapon systems
col_order = ['raion_nominatim_en', 'oblast', 'n_events'] + area_type_vars + weapon_sys_vars
d_lvl2_raion = d_lvl2_raion[col_order]

# Sort by total events (descending)
d_lvl2_raion = d_lvl2_raion.sort_values('n_events', ascending=False).reset_index(drop=True)

# Summary
print(f"Raion-level aggregation: {len(d_lvl2_raion)} raions")
print(f"Total events: {d_lvl2_raion['n_events'].sum():,}")
print(f"\nTop 10 raions by event count:")
d_lvl2_raion.head(10)

Raion-level aggregation: 20 raions
Total events: 88

Top 10 raions by event count:


,raion_nominatim_en,oblast,n_events,admn,cmrc,cltr,food,hlth,hmnt,indl,mltr,rlgs,rsdn,trsp,schl,undf,arst,amsl,blsc,clst,crsm,mrtr,rckt,tube,incd,mine,ltrm,none,smll,thrm,unkn,vhcl
0,Kharkiv Raion,Kharkiv Oblast,30,0,2,0,0,0,0,0,1,0,23,0,3,0,0,0,0,17,0,0,2,0,0,0,0,0,2,0,10,0
1,Bucha Raion,Kyiv Oblast,10,0,1,0,0,1,0,0,0,0,5,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,3,0,4,2
2,Volnovakha Raion,Donetsk Oblast,6,0,2,0,0,2,0,0,0,0,3,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,5,0
3,Donetsk Raion,Donetsk Oblast,6,0,0,0,0,2,0,2,0,0,2,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,4,0
4,Chernihiv Raion,Chernihiv Oblast,5,0,2,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0
5,Mariupol Raion,Donetsk Oblast,5,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,4,0
6,Okhtyrka Raion,Sumy Oblast,4,0,2,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,3,0
7,Starobilsk Raion,Luhansk Oblast,4,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0
8,Horlivka Raion,Donetsk Oblast,3,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,2,0
9,Kherson Raion,Kherson Oblast,3,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,2,0


In [ ]:
#Export raion-level data for 1:n merge
d.to_csv(f'{DATA_PROC}/d_lvl2_raion.csv', index = False) 

In [28]:
# Aggregate at oblast level

### Collapses event-level data to oblast-level counts

# Build aggregation dict: sum all dummy vars, count events
agg_dict_obl = {var: 'sum' for var in all_dummy_vars}
agg_dict_obl['id'] = 'count'

# Aggregate by oblast
d_lvl2_oblast = d.groupby('oblast', as_index=False).agg(agg_dict_obl)

# Rename id count column
d_lvl2_oblast = d_lvl2_oblast.rename(columns={'id': 'n_events'})

# Add raion count per oblast
raion_counts = d.groupby('oblast')['raion_nominatim_en'].nunique()
d_lvl2_oblast['n_raions'] = d_lvl2_oblast['oblast'].map(raion_counts)

# Reorder columns: oblast, n_raions, n_events, area types, weapon systems
col_order_obl = ['oblast', 'n_raions', 'n_events'] + area_type_vars + weapon_sys_vars
d_lvl2_oblast = d_lvl2_oblast[col_order_obl]

# Sort by total events (descending)
d_lvl2_oblast = d_lvl2_oblast.sort_values('n_events', ascending=False).reset_index(drop=True)

# Summary
print(f"Oblast-level aggregation: {len(d_lvl2_oblast)} oblasts")
print(f"Total events: {d_lvl2_oblast['n_events'].sum():,}")
print(f"\nOblast counts:")
d_lvl2_oblast

Oblast-level aggregation: 12 oblasts
Total events: 99

Oblast counts:


,oblast,n_raions,n_events,admn,cmrc,cltr,food,hlth,hmnt,indl,mltr,rlgs,rsdn,trsp,schl,undf,arst,amsl,blsc,clst,crsm,mrtr,rckt,tube,incd,mine,ltrm,none,smll,thrm,unkn,vhcl
0,Kharkiv Oblast,3,32,0,2,0,0,0,0,0,1,0,24,0,3,0,0,0,0,17,0,0,2,0,0,0,0,0,3,0,11,0
1,Donetsk Oblast,4,20,0,2,0,0,4,0,3,0,0,10,0,1,0,0,0,2,1,0,0,2,0,0,0,0,0,0,0,15,0
2,Kyiv Oblast,3,13,0,1,0,0,1,0,0,0,0,5,1,2,0,0,0,0,0,1,0,0,0,0,0,0,0,3,0,7,2
3,Kyiv,0,8,0,0,0,0,1,0,0,0,0,4,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,4,1
4,Luhansk Oblast,1,8,0,1,0,0,0,0,0,0,0,4,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0
5,Chernihiv Oblast,1,5,0,2,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0
6,Kherson Oblast,3,5,0,2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,4,0
7,Sumy Oblast,1,4,0,2,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,3,0
8,Cherkasy Oblast,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,Mykolaiv Oblast,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


In [29]:
#Export oblast-level data for (potential) 1:n merge
d.to_csv(f'{DATA_PROC}/d_lvl2_oblast.csv', index = False) 